# 08 · Crystals: one tube, five lattices

The same Mo tube (35 kV, 1 mA) and the same 2:1 coupled goniometer, with five crystals in the holder: NaCl (the
calibrated anchor of leaflet Fig. 4), LiF, Si (111), calcite (104) and muscovite (002). A `braggsim.crystal.Crystal` is
data (cell, atoms with their Debye-Waller B, cut plane); `scan.expected_rate(..., crystal=X)` takes it. Peak *positions* follow Bragg's law with the real d of each cut and are exact. Peak *heights* come from the kinematic
ideal-mosaic model (|F|²/(V² μ)) scaled to NaCl and multiplied by an extinction factor y per crystal and order: Darwin-Prins
for Si, calcite and muscovite, secondary extinction for LiF (assumed mosaic 0.05°), relative to NaCl's own y (see the last
section; the assumptions are in model/PARAMETERS.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from braggsim.constants import MO_KA_PM
from braggsim.crystal import (
    PRESETS,
    cell_scale,
    d_pm,
    density_g_cm3,
    structure_factor_sq_hkl,
    theta_from_lambda,
)
from braggsim.scan import DEFAULT, coupled_betas, expected_rate, max_order

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
COLOUR = {
    "NaCl": "#1a1a19",
    "LiF": "#2a78d6",
    "Si": "#eb6834",
    "calcite": "#2f9e6e",
    "muscovite": "#8e5bd0",
}

## The table

d is the spacing of the cut series (order n reflects at d/n). θ(Kα) is the glancing angle of Mo Kα (71.08 pm) in
orders 1-3; in COUPLED mode the target sits at β = θ. The two last columns are the order-1 Kα peak height relative to
NaCl, read from the noise-free scan in **counts** (including Lorentz-polarization, dead time and the leak, scatter and
background under the peak), and the same with leak, scatter and background switched off (Bragg signal only).

In [ ]:
def window(c, orders=3):
    th = [float(theta_from_lambda(MO_KA_PM, n, d_pm(c))) for n in range(1, orders + 1)]
    lo = 1.5 if c.name == "muscovite" else th[0] - 1.5
    return coupled_betas(lo, th[-1] + 1.5, 0.02), th


BRAGG_ONLY = DEFAULT.replace(leak_amp_per_s=0.0, scatter_per_s=0.0, background_per_s=0.0)


def peak(c, n=1, half=0.25, params=DEFAULT):
    # Local maximum of R near θ_n(Kα), or NaN where there is none (a monotonic slope).
    t = float(theta_from_lambda(MO_KA_PM, n, d_pm(c)))
    b = coupled_betas(max(t - half, 1.5), t + half, 0.01)
    r = expected_rate(b, 35.0, 1.0, params, crystal=c)
    inner = (r[1:-1] > r[:-2]) & (r[1:-1] >= r[2:])
    return float(r[1:-1][inner].max()) if inner.any() else float("nan")


h_nacl = peak(PRESETS["NaCl"])
b_nacl = peak(PRESETS["NaCl"], params=BRAGG_ONLY)
rows = []
for name, c in PRESETS.items():
    th = [float(theta_from_lambda(MO_KA_PM, n, d_pm(c))) for n in (1, 2, 3)]
    rows.append(
        {
            "crystal": name,
            "cut": str(c.cut),
            "d (pm)": round(d_pm(c), 2),
            "ρ (g/cm³)": round(density_g_cm3(c), 3),
            "θ Kα n=1 (°)": round(th[0], 2),
            "n=2 (°)": round(th[1], 2),
            "n=3 (°)": round(th[2], 2),
            "highest order": max_order(c),
            "cell scale": round(cell_scale(c), 3),
            "Kα n=1 peak (counts/s)": (
                round(peak(c)) if peak(c) == peak(c) else "— (no maximum, on the leak slope)"
            ),
            "counts ratio to NaCl (incl. LP, dead time, background)": (
                round(peak(c) / h_nacl, 2) if peak(c) == peak(c) else "—"
            ),
            "Bragg-only ratio to NaCl": round(peak(c, params=BRAGG_ONLY) / b_nacl, 2),
        }
    )
pd.DataFrame(rows).set_index("crystal")

## Coupled scans, orders 1-3

Each panel covers the first three orders of its crystal (Kβ sits just below each Kα: sin θ scales by 63.10/71.08).
Top: linear, bottom: log. Same tube, same counter; only the lattice changes.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (name, c) in enumerate(PRESETS.items()):
    b, th = window(c)
    r = expected_rate(b, 35.0, 1.0, crystal=c)
    for row, ax in enumerate(axes[:, col]):
        ax.plot(b, r, color=COLOUR[name])
        for n, t in enumerate(th, 1):
            ax.axvline(t, color="#999", lw=0.6, ls=":")
            if row == 0:
                ax.annotate(
                    f"n={n}",
                    (t, 0.97),
                    xycoords=("data", "axes fraction"),
                    ha="center",
                    fontsize=8,
                    color="#666",
                )
        if row:
            ax.set_yscale("log")
            ax.set_xlabel("β (°)")
    axes[0, col].set_title(f"{name}  d = {d_pm(c):.0f} pm", fontsize=10)
axes[0, 0].set_ylabel("R (counts/s)")
axes[1, 0].set_ylabel("R (counts/s), log")
fig.tight_layout()

## Same axes: where the orders land

All five crystals on one θ axis, linear scale, β from 1.5° to 25°: different d moves every peak, and the heights follow
|F|², Lorentz-polarization and the crystal's own absorption.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
b = coupled_betas(1.5, 25.0, 0.02)
for name, c in PRESETS.items():
    ax.plot(b, expected_rate(b, 35.0, 1.0, crystal=c), color=COLOUR[name], label=name)
ax.set_xlabel("β (°)")
ax.set_ylabel("R (counts/s)")
ax.legend(frameon=False, ncols=5)
fig.tight_layout()

## What to notice

- **LiF** has the smallest d of the five (201 pm), so its peaks sit at *larger* θ (Kα n=1 at 10.2°). The Kα doublet
  (70.93/71.36 pm) resolves best there at equal order because its orders sit at larger θ (dθ/dλ = tanθ/λ at fixed n).
  With the same instrument width it is the best-resolved doublet of the set.
- **Why LiF reads 1.1 × NaCl in counts.** The kinematic (y = 1) order-1 reflectivity ratio is several times NaCl's, but
  LiF's low μ means strong secondary extinction (y = 0.31 against NaCl's reference 0.81 at Mo Kα), which brings the
  order-1 Kα line weight to 1.22 × NaCl. Lorentz-polarization is smaller at 10.17° than at 7.24° and the counter's dead
  time (τ = 100 µs) takes more off LiF's tall peak, so the counts ratio is about 1.1. Si and calcite are Darwin-limited
  (y = 0.035 and 0.047): their line weights are 0.033 and 0.034 of NaCl's and the peaks read about 0.047 and 0.048 in counts
  (a larger Lorentz-polarization at smaller θ, and less dead time).
- **Si (111)**: the second order (222) is exactly forbidden for spherical atoms with equal B in the diamond lattice (the
  small real (222) from bonding electrons is not modelled), so the n=2 peak is almost absent while n=3 (333) is strong.
- **Muscovite (002)** has d ≈ 1000 pm, so Kα n=1 falls at 2.04°, buried in the direct-beam leak (a Gaussian of
  σ = 1.16° in 2θ (0.58° in β) around 0°); the clean orders are 2 and 3. The mica's long cell gives many orders: up to order 56 can
  reach the counter at 35 kV. Order 1 has no maximum at all: the scan is on the falling leak
  slope, so the table shows no peak height. Its Bragg-only ratio to NaCl (right column) is about 0.03.
- **Calcite (104)**: |F|² is not monotonic in order (n=2 0.133, n=3 0.169 relative to n=1), but n=2 reads higher
  than n=3 (19 against 13 counts/s, Bragg only) because Lorentz-polarization falls with θ.

## What the heights mean, and what they do not

Peak *positions* are exact: n λ = 2 d sin θ with the cell of each preset. Peak *heights* are the kinematic thick-crystal
value ∝ |F|²/(V² μ), scaled to the fitted NaCl, times y_X,n(λ)/y_NaCl,1(λ), where y ∈ (0, 1] is the extinction factor
(`braggsim.crystal.extinction_factor`, a table on the μ/ρ grid):

- Si, calcite and muscovite are treated as perfect crystals (Darwin-Prins integrated reflectivity). y at Mo Kα, order 1
  is 0.035, 0.047 and 0.165. Muscovite is the least certain: a mosaic sheet (0.005-0.01°) would read 3.6-4.4 × brighter.
- LiF is a mosaic with an *assumed* spread of 0.05° (secondary extinction): y = 0.31 (order 1), 0.77 (order 2, the (400)),
  0.94 (order 3). NaCl's own scan is the Fig. 4 anchor; only the reference y (0.2°, B = 1.3 Å²) enters the ratio.
- Not modelled: finite crystal thickness, primary extinction inside mosaic blocks, beam-footprint overfill at small θ
  (muscovite at 2.04°), Si (222) from bonding charge.
- Order-to-order ratios inside one crystal are more trustworthy than ratios between crystals; the same-tube
  comparison of the order-1 Kα peak against NaCl is the check to make on the bench.

The model keeps the leak, the scatter and the background identical for every crystal; they do not depend on it.


In [ ]:
# Si (200) and (222) are exactly forbidden in the kinematic model with spherical atoms and equal B
si = PRESETS["Si"]
print("Si |F|² (200):", structure_factor_sq_hkl(si, 2, 0, 0))
print("Si |F|² (111):", structure_factor_sq_hkl(si, 1, 1, 1))
print("Si |F|² (222):", structure_factor_sq_hkl(si, 2, 2, 2))